# Listings: Modelo dimensional en estrella

Este notebook toma la capa silver de los anuncios (`data/silver/listings.csv`) y la divide en un
**modelo en estrella**: una tabla de hechos rodeada de cuatro dimensiones. El resultado son cinco CSV en
`data/gold/star/`.

**Grano de la tabla de hechos:** un anuncio en el snapshot de Inside Airbnb (25 de junio al 2 de julio
de 2025). Es una foto de un momento: no hay historia de precio ni de ocupación día a día. Lo temporal
sale de las fechas que trae cada anuncio (alta del anfitrión, primera y última reseña).

```
                      dim_host
                         │
  dim_property ──── fact_listing ──── dim_location
                         │
                      dim_date  (scrape: activa · first_review / last_review: inactivas)
```

| Tabla | Llave | Contenido |
|---|---|---|
| `fact_listing` | `id` | Llaves foráneas, precio, capacidad, disponibilidad, reseñas, calificaciones, ocupación e ingreso estimados, datos del anfitrión y banderas de amenidades |
| `dim_host` | `host_id` | Un registro por anfitrión con su bandera de superhost |
| `dim_property` | `property_key` | `property_type`, `room_type` y `property_group` |
| `dim_location` | `location_key` | Las 16 alcaldías con su centroide |
| `dim_date` | `date_key` | Calendario diario (aaaammdd) que cumple varios roles |

## 1. Carga de datos

In [ ]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

SILVER = Path("../data/silver/listings.csv")
BRONZE = Path("../data/bronze/listings.csv")
STAR = Path("../data/gold/star")
STAR.mkdir(parents=True, exist_ok=True)

if not SILVER.exists():
    raise FileNotFoundError("No existe data/silver/listings.csv: ejecute primero notebook/listings_clean.ipynb")

df = pd.read_csv(SILVER, encoding="utf-8", low_memory=False)
# Bronze solo se usa para la corrección de amen_wifi (variantes "fast wifi"), igual que en el EDA
bronze = pd.read_csv(BRONZE, usecols=["id", "amenities"], low_memory=False)
print(f"Silver cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

## 2. Reglas de limpieza del EDA

El notebook `02_eda_listings.ipynb` aplica sobre silver unas reglas que solo viven en memoria: tipos,
eliminación de `minimum_nights > 365`, corrección de `amen_wifi` y variables derivadas. Se repiten aquí
tal cual para que las cifras del modelo coincidan con las del EDA (26,322 anuncios; 23,057 con precio).

In [ ]:
for col in ["host_since", "first_review", "last_review", "last_scraped"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
for col in ["host_is_superhost", "host_identity_verified", "has_availability", "instant_bookable"]:
    df[col] = df[col].astype("boolean")

# Consistencia: estancias mínimas de más de un año se eliminan; first_review < host_since se marca
df = df[df["minimum_nights"] <= 365].copy().reset_index(drop=True)
df["flag_review_antes_host"] = (df["first_review"] < df["host_since"]).astype("int8")

# amen_wifi: "fast wifi – XX mbps" es el mismo servicio de wifi
amen_listas = (bronze.set_index("id")["amenities"]
               .map(lambda x: [a.strip().lower() for a in ast.literal_eval(x)] if isinstance(x, str) else []))
tiene_wifi = amen_listas.map(lambda l: int(any(a == "wifi" or a.startswith("fast wifi") for a in l)))
df["amen_wifi"] = df["id"].map(tiene_wifi).astype("int8")

# Variables derivadas
df["tiene_precio"] = df["price"].notna().astype("int8")
df["price_per_guest"] = (df["price"] / df["accommodates"]).round(2)
df["host_years"] = ((df["last_scraped"] - df["host_since"]).dt.days / 365.25).round(2)
df["host_segment"] = pd.cut(df["host_listings_count"], bins=[0, 1, 5, 20, np.inf],
                            labels=["1 anuncio", "2-5", "6-20", "21+"])

freq_prop = df["property_type"].value_counts(normalize=True)
tipos_frecuentes = freq_prop[freq_prop >= 0.01].index
df["property_group"] = df["property_type"].where(df["property_type"].isin(tipos_frecuentes), "Otro")

q1, q3 = df["price"].quantile([0.25, 0.75])
lim_inf, lim_sup = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
df["outlier_price_iqr"] = ((df["price"] < lim_inf) | (df["price"] > lim_sup)).astype("int8")

assert len(df) == 26_322 and df["tiene_precio"].sum() == 23_057, "El dataset no coincide con el del EDA"
print(f"Dataset del EDA reconstruido: {len(df):,} anuncios ({df['tiene_precio'].sum():,} con precio)")

## 3. Dimensiones

### 3.1 `dim_date`

Es un solo calendario diario que cubre todas las fechas del dataset, desde el año de la primera alta de
anfitrión hasta el cierre de 2025. La llave `date_key` es un entero `aaaammdd`. La tabla de hechos se
relaciona con él tres veces: por fecha de scrape (relación activa) y por primera y última reseña
(relaciones inactivas, que se usan solo cuando el análisis lo pide).

In [ ]:
def a_date_key(serie):
    # Fecha -> entero aaaammdd; los faltantes quedan nulos
    return (serie.dt.year * 10000 + serie.dt.month * 100 + serie.dt.day).astype("Int64")


cols_fecha = ["host_since", "first_review", "last_review", "last_scraped"]
inicio = df[cols_fecha].min().min().replace(month=1, day=1)
fin = df["last_scraped"].max().replace(month=12, day=31)
fechas = pd.Series(pd.date_range(inicio, fin, freq="D"))

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

dim_date = pd.DataFrame({
    "date_key": a_date_key(fechas),
    "fecha": fechas,
    "anio": fechas.dt.year,
    "trimestre": fechas.dt.quarter,
    "mes_num": fechas.dt.month,
    "mes": fechas.dt.month.map(lambda m: MESES[m - 1]),
    "anio_mes": fechas.dt.strftime("%Y-%m"),
    "dia": fechas.dt.day,
    "dia_semana_num": fechas.dt.dayofweek + 1,
    "dia_semana": fechas.dt.dayofweek.map(lambda d: DIAS[d]),
    "es_fin_de_semana": (fechas.dt.dayofweek >= 5).astype("int8"),
    # Mismo corte de pandemia que la gráfica 16 del EDA
    "periodo": np.select([fechas.dt.year <= 2019, fechas.dt.year <= 2021],
                         ["Pre-pandemia", "Pandemia (2020-2021)"], "Post-pandemia"),
})
print(f"dim_date: {dim_date['fecha'].min():%Y-%m-%d} a {dim_date['fecha'].max():%Y-%m-%d} ({len(dim_date):,} días)")
dim_date.head()

### 3.2 `dim_location`

Las 16 alcaldías de `neighbourhood_cleansed`, que Inside Airbnb asigna a partir de las coordenadas. El
centroide es la mediana de las coordenadas de sus anuncios. `ubicacion_mapa` lleva el nombre completo
(alcaldía, ciudad y país) para que un mapa ubique cada alcaldía sin ambigüedad.

In [ ]:
dim_location = (df.groupby("neighbourhood_cleansed")
                .agg(lat_centro=("latitude", "median"), lon_centro=("longitude", "median"))
                .reset_index().rename(columns={"neighbourhood_cleansed": "alcaldia"})
                .sort_values("alcaldia").reset_index(drop=True))
dim_location.insert(0, "location_key", np.arange(1, len(dim_location) + 1))
dim_location["ubicacion_mapa"] = dim_location["alcaldia"] + ", Ciudad de México, México"
dim_location

### 3.3 `dim_property`

Un mismo `property_type` puede aparecer con distintos `room_type`, así que la llave es la combinación
de ambos. `property_group` es la agrupación del EDA: los tipos con menos de 1% de anuncios quedan en
"Otro".

In [ ]:
dim_property = (df[["property_type", "room_type", "property_group"]].drop_duplicates()
                .sort_values(["room_type", "property_type"]).reset_index(drop=True))
dim_property.insert(0, "property_key", np.arange(1, len(dim_property) + 1))
print(f"dim_property: {len(dim_property)} combinaciones de {df['property_type'].nunique()} property_type")
dim_property.head(10)

### 3.4 `dim_host`

Es el *distinct* de `host_id` con una sola bandera: `es_superhost` vale 1 si el anuncio trae
`host_is_superhost = True` y 0 en cualquier otro caso, incluido el vacío. Los demás datos del anfitrión
(antigüedad, tipo de respuesta, tasas, verificaciones) se quedan en la tabla de hechos.

In [ ]:
dim_host = df[["host_id", "host_is_superhost"]].drop_duplicates("host_id").reset_index(drop=True)
dim_host["es_superhost"] = dim_host.pop("host_is_superhost").fillna(False).astype("int8")
print(f"dim_host: {len(dim_host):,} anfitriones")
dim_host.head()

## 4. Tabla de hechos `fact_listing`

Cada anuncio recibe las llaves de sus dimensiones. Se quedan directamente en los hechos, sin dimensión
propia, el nombre del anuncio, las coordenadas, los datos del anfitrión y las banderas 0/1: las 10 amenidades `amen_*`,
`instant_bookable`, `has_availability` y las banderas de calidad del EDA.

In [ ]:
fact = df.copy()
fact["location_key"] = fact["neighbourhood_cleansed"].map(dict(zip(dim_location["alcaldia"], dim_location["location_key"])))
fact = fact.merge(dim_property, on=["property_type", "room_type", "property_group"], how="left", validate="m:1")
fact["scrape_date_key"] = a_date_key(fact["last_scraped"])
fact["first_review_date_key"] = a_date_key(fact["first_review"])
fact["last_review_date_key"] = a_date_key(fact["last_review"])
fact["has_availability"] = fact["has_availability"].astype("Int8")
fact["instant_bookable"] = fact["instant_bookable"].astype("int8")
fact["host_since_year"] = fact["host_since"].dt.year.astype("Int64")
fact["host_response_category"] = fact["host_response_category"].fillna("Sin historial")

cols_fact = (
    ["id", "name", "host_id", "location_key", "property_key",
     "scrape_date_key", "first_review_date_key", "last_review_date_key", "latitude", "longitude",
     "price", "price_per_guest", "accommodates", "bathrooms", "bedrooms", "beds", "amenities_count",
     "minimum_nights", "maximum_nights", "availability_30", "availability_90", "availability_365",
     "number_of_reviews", "number_of_reviews_ltm", "reviews_per_month"]
    + [c for c in df.columns if c.startswith("review_scores_")]
    + ["estimated_occupancy_l365d", "estimated_revenue_l365d"]
    + ["host_since", "host_since_year", "host_years", "host_identity_verified", "host_response_category",
       "host_response_rate", "host_acceptance_rate", "host_listings_count", "host_segment",
       "verif_email", "verif_phone", "verif_work_email"]
    + sorted(c for c in df.columns if c.startswith("amen_"))
    + ["instant_bookable", "has_availability", "tiene_precio", "outlier_price_iqr", "flag_review_antes_host"]
)
fact_listing = fact[cols_fact]
print(f"fact_listing: {fact_listing.shape[0]:,} filas x {fact_listing.shape[1]} columnas")
fact_listing.head()

## 5. Validación del modelo

- Cada llave primaria debe ser única.
- Toda llave foránea de los hechos debe existir en su dimensión (integridad referencial).
- Los totales deben cuadrar con el dataset del EDA.

In [ ]:
dimensiones = {"dim_date": ("date_key", dim_date), "dim_location": ("location_key", dim_location),
               "dim_property": ("property_key", dim_property), "dim_host": ("host_id", dim_host)}
foraneas = {"host_id": "dim_host", "location_key": "dim_location", "property_key": "dim_property",
            "scrape_date_key": "dim_date",
            "first_review_date_key": "dim_date", "last_review_date_key": "dim_date"}

filas = []
for fk, dim in foraneas.items():
    pk, tabla = dimensiones[dim]
    valores = fact_listing[fk].dropna()
    filas.append({"llave foránea": fk, "dimensión": dim, "nulos en hechos": int(fact_listing[fk].isna().sum()),
                  "huérfanas": int((~valores.isin(tabla[pk])).sum())})
integridad = pd.DataFrame(filas).set_index("llave foránea")
display(integridad)

assert fact_listing["id"].is_unique
assert all(tabla[pk].is_unique for pk, tabla in dimensiones.values())
assert integridad["huérfanas"].sum() == 0
assert fact_listing[["host_id", "location_key", "property_key", "scrape_date_key"]].notna().all().all()
assert len(fact_listing) == len(df) and np.isclose(fact_listing["price"].sum(), df["price"].sum())
print("Modelo válido: llaves únicas, sin huérfanas y totales iguales al dataset del EDA")

## 6. Exportación

Los CSV se escriben en UTF-8, con punto decimal y fechas ISO (`aaaa-mm-dd`). Igual que el resto de
`data/gold`, no se versionan: se regeneran ejecutando este notebook.

In [ ]:
tablas = {"fact_listing": fact_listing, "dim_host": dim_host, "dim_property": dim_property,
          "dim_location": dim_location, "dim_date": dim_date}
for nombre, tabla in tablas.items():
    tabla.to_csv(STAR / f"{nombre}.csv", index=False, encoding="utf-8", date_format="%Y-%m-%d")

pd.DataFrame({"filas": {n: len(t) for n, t in tablas.items()},
              "columnas": {n: t.shape[1] for n, t in tablas.items()}}).rename_axis("tabla")